In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print(" Ready")

 Ready


In [2]:
%%writefile agents/web_search_agent.py
import time
from typing import Dict, List

from tools.llm_clients import research_with_fallback
from tools.logger import log_agent_event

AGENT_NAME = "web_search_agent"
MAX_QUERIES = 5


def _dedupe_sources(sources: List[Dict]) -> List[Dict]:
    seen, unique = set(), []
    for s in sources:
        url = s.get("url")
        if url and url not in seen:
            seen.add(url)
            unique.append(s)
    return unique


def search_one(sub_question: str) -> Dict:
    start = time.time()
    try:
        raw = research_with_fallback(sub_question)
        sources = _dedupe_sources(raw.get("sources", []))
        answer = (raw.get("answer") or "").strip()
        status = "ok" if (answer and sources) else "empty"
        result = {
            "sub_question": sub_question, "answer": answer, "sources": sources,
            "provider": raw.get("provider", "unknown"), "status": status,
            "latency_s": round(time.time() - start, 2),
        }
    except Exception as e:
        result = {
            "sub_question": sub_question, "answer": "", "sources": [], "provider": "none",
            "status": "failed", "error": f"{type(e).__name__}: {e}",
            "latency_s": round(time.time() - start, 2),
        }
    log_agent_event(AGENT_NAME, "search_done", status=result["status"],
                    provider=result["provider"], sources=len(result["sources"]))
    return result


def web_search_node(state: Dict) -> Dict:
    seen, queries = set(), []
    for q in state.get("plan", []):
        q = q.strip()
        if q and q.lower() not in seen:
            seen.add(q.lower())
            queries.append(q)
    queries = queries[:MAX_QUERIES]

    if not queries:
        return {
            "errors": [{"agent": AGENT_NAME, "error": "empty plan: nothing to search"}],
            "model_usage_log": [{"agent": AGENT_NAME, "status": "skipped_no_input"}],
        }

    results = [search_one(q) for q in queries]
    good = [r for r in results if r["status"] != "failed"]
    failed = [r for r in results if r["status"] == "failed"]

    usage: Dict[str, int] = {}
    for r in good:
        usage[r["provider"]] = usage.get(r["provider"], 0) + 1

    return {
        "search_results": good,
        "errors": [{"agent": AGENT_NAME, "sub_question": r["sub_question"], "error": r["error"]}
                   for r in failed],
        "model_usage_log": [{"agent": AGENT_NAME, "provider": p, "queries": n}
                            for p, n in usage.items()] or [{"agent": AGENT_NAME, "status": "ran_no_results"}],
    }

Overwriting agents/web_search_agent.py


In [3]:
%%writefile agents/data_extraction_agent.py
import json
import re
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "data_extraction_agent"


def _build_extraction_prompt(sub_question: str, answer: str) -> str:
    return f"""Extract discrete, atomic factual claims from the text below, related to the
question: "{sub_question}"

Rules:
- Each fact must be a single, self-contained, checkable statement.
- Do NOT include opinions, vague generalities, or marketing language.
- If the text contains no clear factual claims, return an empty list.
- Respond with ONLY a JSON array of strings, nothing else.

Text:
\"\"\"{answer}\"\"\"

Example valid response: ["Fact one stated plainly.", "Fact two stated plainly."]
Example for irrelevant/no facts: []
"""


def _parse_json_facts(raw_text: str) -> List[str]:
    cleaned = re.sub(r"^```(json)?", "", raw_text.strip()).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, list):
            return [str(f).strip() for f in parsed if str(f).strip()]
        return []
    except json.JSONDecodeError:
        return []


def extract_facts_from_result(search_result: Dict) -> List[Dict]:
    sub_question = search_result.get("sub_question", "")
    answer = search_result.get("answer", "")
    sources = search_result.get("sources", [])

    if not answer.strip():
        return []

    try:
        raw_output = call_llm(_build_extraction_prompt(sub_question, answer))
        raw_facts = _parse_json_facts(raw_output)
    except Exception as e:
        log_agent_event(AGENT_NAME, "extraction_failed", sub_question=sub_question,
                        error=f"{type(e).__name__}: {e}")
        return []

    facts = [{"claim": t, "sub_question": sub_question, "sources": sources} for t in raw_facts]
    log_agent_event(AGENT_NAME, "extraction_done", sub_question=sub_question, facts_found=len(facts))
    return facts


def data_extraction_node(state: Dict) -> Dict:
    search_results = state.get("search_results", [])
    if not search_results:
        return {
            "errors": [{"agent": AGENT_NAME, "error": "no search_results to extract from"}],
            "model_usage_log": [{"agent": AGENT_NAME, "status": "skipped_no_input"}],
        }

    all_facts: List[Dict] = []
    errors: List[Dict] = []
    for result in search_results:
        try:
            all_facts.extend(extract_facts_from_result(result))
        except Exception as e:
            errors.append({"agent": AGENT_NAME, "sub_question": result.get("sub_question", "unknown"),
                          "error": f"{type(e).__name__}: {e}"})

    return {
        "extracted_facts": all_facts,
        "errors": errors,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "calls": len(search_results)}],
    }

Overwriting agents/data_extraction_agent.py


In [4]:
%%writefile agents/fact_checking_agent.py
import json
import re
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "fact_checking_agent"
MAX_FACTS_PER_BATCH = 15


def _build_fact_check_prompt(facts: List[Dict]) -> str:
    numbered = "\n".join(f'{i}. "{f["claim"]}" (sources: {len(f.get("sources", []))})'
                         for i, f in enumerate(facts))
    return f"""You are fact-checking a set of claims gathered during research. For EACH
numbered claim below, classify it as exactly one of:

- CONFIRMED: corroborated by multiple independent sources, or clearly consistent with other claims
- DISPUTED: contradicts another claim in this list
- UNVERIFIED: plausible but only backed by a single source, with no corroboration or contradiction

Claims:
{numbered}

Respond with ONLY a JSON array: [{{"index": 0, "status": "CONFIRMED", "reasoning": "..."}}, ...]
"""


def _parse_fact_check_response(raw_text: str, expected_count: int) -> List[Dict]:
    cleaned = re.sub(r"^```(json)?", "", raw_text.strip()).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
        if not isinstance(parsed, list):
            return []
        results = {}
        for item in parsed:
            idx = item.get("index")
            if isinstance(idx, int) and 0 <= idx < expected_count:
                status = item.get("status", "UNVERIFIED").upper()
                if status not in ("CONFIRMED", "DISPUTED", "UNVERIFIED"):
                    status = "UNVERIFIED"
                results[idx] = {"status": status, "reasoning": item.get("reasoning", "")}
        return [results.get(i, {"status": "UNVERIFIED", "reasoning": "not returned by model"})
                for i in range(expected_count)]
    except json.JSONDecodeError:
        return []


def fact_check_batch(facts: List[Dict]) -> List[Dict]:
    if not facts:
        return []
    try:
        raw_output = call_llm(_build_fact_check_prompt(facts))
        verdicts = _parse_fact_check_response(raw_output, len(facts))
        if not verdicts:
            raise ValueError("fact-check response could not be parsed")
    except Exception as e:
        log_agent_event(AGENT_NAME, "batch_failed", error=f"{type(e).__name__}: {e}", batch_size=len(facts))
        verdicts = [{"status": "UNVERIFIED", "reasoning": "fact-check call failed; defaulted to unverified"}
                   for _ in facts]

    checked = [{**fact, "status": v["status"], "reasoning": v["reasoning"]} for fact, v in zip(facts, verdicts)]
    counts = {}
    for v in verdicts:
        counts[v["status"]] = counts.get(v["status"], 0) + 1
    log_agent_event(AGENT_NAME, "batch_done", **counts)
    return checked


def fact_check_facts(facts: List[Dict]) -> List[Dict]:
    all_checked = []
    for i in range(0, len(facts), MAX_FACTS_PER_BATCH):
        all_checked.extend(fact_check_batch(facts[i:i + MAX_FACTS_PER_BATCH]))
    return all_checked


def fact_checking_node(state: Dict) -> Dict:
    extracted_facts = state.get("extracted_facts", [])
    if not extracted_facts:
        return {
            "errors": [{"agent": AGENT_NAME, "error": "no extracted_facts to check"}],
            "model_usage_log": [{"agent": AGENT_NAME, "status": "skipped_no_input"}],
        }

    checked = fact_check_facts(extracted_facts)
    return {
        "fact_check_results": checked,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "facts_checked": len(checked)}],
    }

Overwriting agents/fact_checking_agent.py


In [5]:

%%writefile agents/summarization_agent.py
from collections import defaultdict
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "summarization_agent"


def _group_by_sub_question(fact_check_results: List[Dict]) -> Dict[str, List[Dict]]:
    grouped = defaultdict(list)
    for fact in fact_check_results:
        grouped[fact.get("sub_question", "unknown")].append(fact)
    return dict(grouped)


def _build_summary_prompt(sub_question: str, facts: List[Dict]) -> str:
    confirmed = [f for f in facts if f["status"] == "CONFIRMED"]
    disputed = [f for f in facts if f["status"] == "DISPUTED"]
    unverified = [f for f in facts if f["status"] == "UNVERIFIED"]

    def fmt(fs):
        return "\n".join(f'- "{f["claim"]}" (reasoning: {f.get("reasoning", "")})' for f in fs) or "(none)"

    return f"""Write a short, honest research brief (3-6 sentences) answering: "{sub_question}"

Rules:
- State CONFIRMED facts plainly.
- For DISPUTED facts, explicitly say they are disputed and describe BOTH sides.
- For UNVERIFIED facts, hedge explicitly (e.g. "according to a single source...").
- Do not invent facts. Do not hide disputes.

CONFIRMED facts:
{fmt(confirmed)}

DISPUTED facts:
{fmt(disputed)}

UNVERIFIED facts:
{fmt(unverified)}

Write only the brief itself.
"""


def summarize_sub_topic(sub_question: str, facts: List[Dict]) -> Dict:
    if not facts:
        return {
            "sub_question": sub_question,
            "summary": "No verified information available for this sub-topic.",
            "status": "no_facts",
            "fact_counts": {"CONFIRMED": 0, "DISPUTED": 0, "UNVERIFIED": 0},
        }

    counts = {"CONFIRMED": 0, "DISPUTED": 0, "UNVERIFIED": 0}
    for f in facts:
        counts[f["status"]] = counts.get(f["status"], 0) + 1

    try:
        summary_text = call_llm(_build_summary_prompt(sub_question, facts)).strip()
        status = "ok"
    except Exception as e:
        summary_text = f"Summary could not be generated due to an error ({type(e).__name__}). Raw fact counts: {counts}."
        status = "failed"
        log_agent_event(AGENT_NAME, "summary_failed", sub_question=sub_question, error=f"{type(e).__name__}: {e}")

    log_agent_event(AGENT_NAME, "summary_done", sub_question=sub_question, status=status, **counts)
    return {"sub_question": sub_question, "summary": summary_text, "status": status, "fact_counts": counts}


def summarization_node(state: Dict) -> Dict:
    fact_check_results = state.get("fact_check_results", [])
    if not fact_check_results:
        return {
            "errors": [{"agent": AGENT_NAME, "error": "no fact_check_results to summarize"}],
            "model_usage_log": [{"agent": AGENT_NAME, "status": "skipped_no_input"}],
        }

    grouped = _group_by_sub_question(fact_check_results)
    summaries = [summarize_sub_topic(q, facts) for q, facts in grouped.items()]
    return {
        "summaries": summaries,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "sub_topics": len(summaries)}],
    }

Overwriting agents/summarization_agent.py


In [6]:
%%writefile agents/report_generator_agent.py
from typing import Dict, List
from collections import Counter

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "report_generator_agent"


def _build_narrative_prompt(query: str, summaries: List[Dict]) -> str:
    sections = "\n\n".join(f"## {s['sub_question']}\n{s['summary']}" for s in summaries)
    return f"""Write a clear, well-organized research report answering: "{query}"

Combine the sub-topic briefs below into flowing prose with headings. Do not invent facts.
Preserve hedging language exactly as given — do not make claims sound more certain.

Sub-topic briefs:
{sections}

Markdown, short intro, one section per sub-topic, one-paragraph conclusion.
Do not include a confidence/sources appendix — that is added separately.
"""


def _generate_narrative(query: str, summaries: List[Dict]) -> Dict:
    try:
        text = call_llm(_build_narrative_prompt(query, summaries)).strip()
        return {"text": text, "method": "llm"}
    except Exception as e:
        log_agent_event(AGENT_NAME, "narrative_failed", error=f"{type(e).__name__}: {e}")
        fallback = f"# Research Report: {query}\n\n" + "\n\n".join(
            f"## {s['sub_question']}\n{s['summary']}" for s in summaries)
        return {"text": fallback, "method": "fallback_template"}


def _build_confidence_appendix(fact_check_results: List[Dict]) -> str:
    if not fact_check_results:
        return "\n\n---\n## Confidence Appendix\nNo fact-checked claims were available.\n"

    counts = Counter(f["status"] for f in fact_check_results)
    total = len(fact_check_results)
    lines = ["\n\n---\n## Confidence Appendix\n"]
    lines.append(f"**Total claims reviewed:** {total}  ")
    lines.append(f"**Confirmed:** {counts.get('CONFIRMED', 0)} | "
                f"**Disputed:** {counts.get('DISPUTED', 0)} | "
                f"**Unverified:** {counts.get('UNVERIFIED', 0)}\n")

    disputed = [f for f in fact_check_results if f["status"] == "DISPUTED"]
    if disputed:
        lines.append("### ⚠️ Disputed claims\n")
        for f in disputed:
            lines.append(f"- \"{f['claim']}\" — *{f.get('reasoning', '')}*")

    unverified = [f for f in fact_check_results if f["status"] == "UNVERIFIED"]
    if unverified:
        lines.append("\n### ❓ Unverified claims (single source, no corroboration)\n")
        for f in unverified:
            lines.append(f"- \"{f['claim']}\"")

    return "\n".join(lines)


def generate_report(query: str, summaries: List[Dict], fact_check_results: List[Dict]) -> Dict:
    if not summaries:
        return {
            "report": f"# Research Report: {query}\n\nNo verified information was found for this query.",
            "method": "no_data",
        }

    narrative = _generate_narrative(query, summaries)
    appendix = _build_confidence_appendix(fact_check_results)
    full_report = narrative["text"] + appendix

    log_agent_event(AGENT_NAME, "report_done", method=narrative["method"],
                    sub_topics=len(summaries), total_claims=len(fact_check_results))
    return {"report": full_report, "method": narrative["method"]}


def report_generator_node(state: Dict) -> Dict:
    summaries = state.get("summaries", [])
    fact_check_results = state.get("fact_check_results", [])
    query = state.get("query", "Unknown query")

    if not summaries:
        return {
            "errors": [{"agent": AGENT_NAME, "error": "no summaries to build report from"}],
            "model_usage_log": [{"agent": AGENT_NAME, "status": "skipped_no_input"}],
        }

    result = generate_report(query, summaries, fact_check_results)
    return {
        "final_report": result["report"],
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "method": result["method"]}],
    }

Overwriting agents/report_generator_agent.py


In [7]:
%%writefile agents/supervisor_agent.py
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "supervisor_agent"
MAX_ITERATIONS = 8

# Hard data-dependency order: each stage needs the previous stage's output to function.
# This is NOT an LLM decision — it's fixed by what data exists at each point, so routing
# it deterministically is more reliable, cheaper, and fully testable without mocking.
PIPELINE = [
    ("web_search_agent", "web_search"),
    ("data_extraction_agent", "data_extraction"),
    ("fact_checking_agent", "fact_checking"),
    ("summarization_agent", "summarization"),
    ("report_generator_agent", "report_generator"),
]


def _build_plan_prompt(query: str) -> str:
    return f"""Break this research question into 2-5 specific, researchable sub-questions:
"{query}"

Respond with each sub-question on its own line, nothing else. No numbering, no preamble.
"""


def generate_plan(query: str) -> List[str]:
    """The one genuinely ambiguous decision the Supervisor makes with an LLM —
    everything else below is deterministic rule-based routing."""
    try:
        raw = call_llm(_build_plan_prompt(query))
        sub_questions = [line.strip("- ").strip() for line in raw.split("\n") if line.strip()]
        return sub_questions[:5] if sub_questions else [query]
    except Exception as e:
        log_agent_event(AGENT_NAME, "plan_generation_failed", error=f"{type(e).__name__}: {e}")
        return [query]  # fail-safe: treat the raw query as the only sub-question


def _agents_completed(state: Dict) -> set:
    """Reads model_usage_log to determine which agents have already been attempted
    (success OR a logged 'ran but found nothing' / 'skipped, no input' marker)."""
    return {entry.get("agent") for entry in state.get("model_usage_log", []) if entry.get("agent")}


def supervisor_node(state: Dict) -> Dict:
    """Deterministic router. Only calls an LLM once, to generate the initial plan."""
    iteration = state.get("iteration_count", 0) + 1

    if iteration > MAX_ITERATIONS:
        log_agent_event(AGENT_NAME, "max_iterations_exceeded", iteration=iteration)
        return {"iteration_count": iteration, "next_agent": "failed"}

    updates: Dict = {"iteration_count": iteration}

    if not state.get("plan"):
        plan = generate_plan(state["query"])
        updates["plan"] = plan
        updates["next_agent"] = "web_search"
        log_agent_event(AGENT_NAME, "plan_generated", sub_questions=len(plan))
        return updates

    completed = _agents_completed(state)
    for agent_name, node_name in PIPELINE:
        if agent_name not in completed:
            updates["next_agent"] = node_name
            log_agent_event(AGENT_NAME, "routing", decision=node_name, iteration=iteration)
            return updates

    updates["next_agent"] = "done"
    log_agent_event(AGENT_NAME, "pipeline_complete", iteration=iteration)
    return updates


def route_from_supervisor(state: Dict) -> str:
    """Conditional-edge function: just reads what supervisor_node already decided."""
    return state.get("next_agent", "failed")

Writing agents/supervisor_agent.py


In [8]:
%%writefile tests/test_supervisor_agent.py
import agents.supervisor_agent as sup


def test_plan_generation_fallback_on_failure(monkeypatch):
    def boom(p):
        raise RuntimeError("down")
    monkeypatch.setattr(sup, "call_llm", boom)
    plan = sup.generate_plan("my query")
    assert plan == ["my query"]


def test_routes_to_planning_when_no_plan(monkeypatch):
    monkeypatch.setattr(sup, "generate_plan", lambda q: ["sub-q 1", "sub-q 2"])
    state = {"query": "q", "iteration_count": 0, "plan": [], "model_usage_log": []}
    out = sup.supervisor_node(state)
    assert out["next_agent"] == "web_search"
    assert out["plan"] == ["sub-q 1", "sub-q 2"]


def test_routes_through_full_pipeline_in_order():
    pipeline_agents = [a for a, _ in sup.PIPELINE]
    state = {"query": "q", "iteration_count": 0, "plan": ["x"], "model_usage_log": []}

    for i, (agent_name, node_name) in enumerate(sup.PIPELINE):
        out = sup.supervisor_node(state)
        assert out["next_agent"] == node_name, f"expected {node_name} at step {i}, got {out['next_agent']}"
        state["model_usage_log"].append({"agent": agent_name})
        state["iteration_count"] = out["iteration_count"]

    final = sup.supervisor_node(state)
    assert final["next_agent"] == "done"


def test_loop_guard_triggers():
    state = {"query": "q", "iteration_count": sup.MAX_ITERATIONS, "plan": ["x"], "model_usage_log": []}
    out = sup.supervisor_node(state)
    assert out["next_agent"] == "failed"


def test_route_from_supervisor_reads_next_agent():
    state = {"next_agent": "fact_checking"}
    assert sup.route_from_supervisor(state) == "fact_checking"

Writing tests/test_supervisor_agent.py


In [ ]:
from graph.build_graph import build_graph

app = build_graph()

initial_state = {
    "query": "What are the main safety risks of solid-state EV batteries?",
    "plan": [], "next_agent": "", "iteration_count": 0, "needs_more_research": False,
    "search_results": [], "extracted_facts": [], "fact_check_results": [], "summaries": [],
    "errors": [], "final_report": None, "model_usage_log": [],
}

result = app.invoke(initial_state)

print("Iterations used:", result["iteration_count"])
print("Plan:", result["plan"])
print("Errors:", result["errors"])
print("\n" + "="*60)
print(result["final_report"])

ModuleNotFoundError: No module named 'graph.build_graph'

In [3]:
import sys, os
print("Current directory:", os.getcwd())
sys.path.append(".")

print("graph/ exists:", os.path.isdir("graph"))
print("graph/build_graph.py exists:", os.path.isfile("graph/build_graph.py"))
print("graph/__init__.py exists:", os.path.isfile("graph/__init__.py"))

Current directory: c:\Users\RIDDHI ASHAR\Desktop\multi-agent-research-system
graph/ exists: True
graph/build_graph.py exists: False
graph/__init__.py exists: True


In [4]:
%%writefile graph/build_graph.py
from langgraph.graph import StateGraph, START, END
from state.schema import ResearchState
from agents.supervisor_agent import supervisor_node, route_from_supervisor
from agents.web_search_agent import web_search_node
from agents.data_extraction_agent import data_extraction_node
from agents.fact_checking_agent import fact_checking_node
from agents.summarization_agent import summarization_node
from agents.report_generator_agent import report_generator_node


def build_graph():
    graph = StateGraph(ResearchState)

    graph.add_node("supervisor", supervisor_node)
    graph.add_node("web_search", web_search_node)
    graph.add_node("data_extraction", data_extraction_node)
    graph.add_node("fact_checking", fact_checking_node)
    graph.add_node("summarization", summarization_node)
    graph.add_node("report_generator", report_generator_node)

    graph.add_edge(START, "supervisor")
    graph.add_conditional_edges("supervisor", route_from_supervisor, {
        "web_search": "web_search",
        "data_extraction": "data_extraction",
        "fact_checking": "fact_checking",
        "summarization": "summarization",
        "report_generator": "report_generator",
        "done": END,
        "failed": END,
    })
    for node in ["web_search", "data_extraction", "fact_checking", "summarization", "report_generator"]:
        graph.add_edge(node, "supervisor")

    return graph.compile()

Writing graph/build_graph.py


In [5]:
import os
print("graph/build_graph.py exists:", os.path.isfile("graph/build_graph.py"))

graph/build_graph.py exists: True


In [6]:
from graph.build_graph import build_graph
app = build_graph()
print(" Graph built successfully")

 Graph built successfully
